# 02 - Extração: SQL Server → MinIO (CSV)

Extrai **todas as tabelas** do database `Ecommerce` do SQL Server e exporta como CSV para o bucket `landing-zone` no MinIO.

**Pré-requisitos:** Docker Compose rodando, Notebook `00` e `01` executado.

## 1. Configuração

In [1]:
import os, io
import pandas as pd
import pyodbc
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')

SQL Server: localhost:1433/Ecommerce
MinIO: http://localhost:9020 | Bucket: landing-zone


## 2. Conexão com SQL Server

In [2]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;'
)
cursor = conn.cursor()
print(f'Conectado ao SQL Server [{DB_DATABASE}]')

Conectado ao SQL Server [Ecommerce]


## 3. Listar Tabelas

In [3]:
cursor.execute("""
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = 'dbo'
    ORDER BY TABLE_NAME
""")
tabelas = [row[0] for row in cursor.fetchall()]

print(f'{len(tabelas)} tabelas encontradas:')
for i, t in enumerate(tabelas, 1):
    cursor.execute(f'SELECT COUNT(*) FROM [{t}]')
    print(f'  {i:2d}. {t:<20} ({cursor.fetchone()[0]:>6} registros)')

5 tabelas encontradas:
   1. categorias           (     5 registros)
   2. clientes             (    10 registros)
   3. itens_venda          (    13 registros)
   4. produtos             (    10 registros)
   5. vendas               (    10 registros)


## 4. Criar Bucket no MinIO

In [4]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] já existe')
except:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [landing-zone] criado!
Buckets: ['landing-zone']


## 5. Extrair Tabelas → CSV no MinIO

In [5]:
print(f'Extraindo {len(tabelas)} tabelas...\n')
resultados = []

for tabela in tabelas:
    # Ler tabela via cursor (evita warning do pd.read_sql com pyodbc)
    cursor.execute(f'SELECT * FROM [{tabela}]')
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=columns)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    csv_bytes = csv_buffer.getvalue().encode('utf-8')
    s3_key = f'{tabela}.csv'

    s3_client.put_object(
        Bucket=LANDING_BUCKET, Key=s3_key,
        Body=csv_bytes, ContentType='text/csv'
    )

    size_kb = len(csv_bytes) / 1024
    resultados.append({'tabela': tabela, 'registros': len(df), 'colunas': len(df.columns), 'tamanho_kb': round(size_kb,1)})
    print(f'  {tabela}.csv -> s3://{LANDING_BUCKET}/{s3_key} ({len(df)} registros, {size_kb:.1f} KB)')

print(f'\nExtracao concluida! {len(tabelas)} tabelas exportadas.')

Extraindo 5 tabelas...

  categorias.csv -> s3://landing-zone/categorias.csv (5 registros, 0.3 KB)
  clientes.csv -> s3://landing-zone/clientes.csv (10 registros, 0.3 KB)
  itens_venda.csv -> s3://landing-zone/itens_venda.csv (13 registros, 0.2 KB)
  produtos.csv -> s3://landing-zone/produtos.csv (10 registros, 0.3 KB)
  vendas.csv -> s3://landing-zone/vendas.csv (10 registros, 0.3 KB)

Extracao concluida! 5 tabelas exportadas.


## 6. Validação

In [6]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in response.get('Contents', []):
    print(f'  {obj["Key"]:<25} {obj["Size"]/1024:>8.1f} KB')

df_resumo = pd.DataFrame(resultados)
print(f'\nTotal: {df_resumo["registros"].sum():,} registros | {df_resumo["tamanho_kb"].sum():,.1f} KB')

Arquivos no bucket [landing-zone]:

  categorias.csv                 0.3 KB
  clientes.csv                   0.3 KB
  itens_venda.csv                0.2 KB
  produtos.csv                   0.3 KB
  vendas.csv                     0.3 KB

Total: 48 registros | 1.4 KB


In [7]:
cursor.close()
conn.close()
print('Conexao SQL Server encerrada.')

Conexao SQL Server encerrada.
